# E8^k HAWK-Style Signing Stage

This notebook works as follows: hash `(message, salt)` to a binary challenge `h in {0,1}^{8k}`, replace HAWK's ambient `Z^{2n}` basis by a direct sum basis `B_E8` of `E8^k`, form the half lattice target `(h * B_E8) / 2`, and use the `b = 2` bootstrap sampler. To sample the half shift at width `sigma`, the helper samples an integer representative from `(h * B_E8) + 2E8^k` at width `2 * sigma` and then halves it.

The returned signature coordinates satisfy `w = h - 2s`, where `w * B_E8` is the sampled integer representative. This is still an experimental E8 signing stage, not a full HAWK implementation.


In [1]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler as DGaussZ
from sage.all import vector, matrix, ZZ, QQ, codes
from mpmath import mp, exp, jtheta, sqrt, pi, log

import itertools
import random
import hashlib
import os

mp.dps = 80

In [ ]:
C = codes.BinaryReedMullerCode(1, 3)
CODEWORD_TUPLES = [tuple(int(x) for x in cw) for cw in C]
CODEWORD_SET = set(CODEWORD_TUPLES)

# Construction A setup for the E8 model used by sampler:
# E8 = RM(1,3) + 2 Z^8, up to scaling.
I2 = [vector(ZZ, [2 if i == j else 0 for j in range(8)]) for i in range(8)]
G = C.generator_matrix()
Grows = [vector(ZZ, [int(x) for x in G.row(i)]) for i in range(G.nrows())]
E8_BASIS = matrix(ZZ, I2 + Grows).row_module().basis_matrix()
E8_PUBLIC_GRAM_BLOCK = E8_BASIS * E8_BASIS.transpose()


def e8_mod_2e8_representatives():
    reps = []
    for bits in itertools.product([0, 1], repeat=8):
        r = vector(ZZ, [0] * 8)
        for i, b in enumerate(bits):
            if b:
                r += E8_BASIS.row(i)
        reps.append(r)
    return reps


def direct_sum_e8_basis(k):
    if k <= 0:
        raise ValueError("k must be positive")
    dim = E8_BASIS.nrows()
    basis = matrix(ZZ, dim * k, dim * k)
    for block in range(k):
        offset = block * dim
        for i in range(dim):
            for j in range(dim):
                basis[offset + i, offset + j] = E8_BASIS[i, j]
    return basis


def public_key_gram_matrix(k, basis=None):
    if basis is None:
        basis = direct_sum_e8_basis(k)
    return basis * basis.transpose()


def serialize_integer_matrix(mat):
    rows = []
    for i in range(mat.nrows()):
        rows.append(",".join(str(ZZ(mat[i, j])) for j in range(mat.ncols())))
    return (";".join(rows)).encode("ascii")


def public_key_gram_descriptor(k):
    # For E8^k this describes the block diagonal public Gram matrix.
    return (
        b"E8-direct-sum-gram-v1"
        + int(k).to_bytes(4, "big")
        + serialize_integer_matrix(E8_PUBLIC_GRAM_BLOCK)
    )


REPS = e8_mod_2e8_representatives()
print("single-block quotient size:", len(REPS))
print("E8 basis shape:", E8_BASIS.nrows(), "x", E8_BASIS.ncols())
print("single-block public Gram shape:", E8_PUBLIC_GRAM_BLOCK.nrows(), "x", E8_PUBLIC_GRAM_BLOCK.ncols())


single-block quotient size: 256
E8 basis shape: 8 x 8
single-block public Gram shape: 8 x 8


In [3]:
def nome(s):
    return exp(-pi / (s**2))


def rho_2Z_shift(alpha, s):
    q = nome(s)
    z = -2j * pi * alpha / (s**2)
    value = (q ** (alpha**2)) * jtheta(3, z, q**4)
    return float(value.real)


def codeword_weights_shifted_E8(s, shift):
    shift_f = [float(a) for a in shift]
    log_weights = []
    for c in CODEWORD_TUPLES:
        lw = 0.0
        for i in range(8):
            lw += float(log(rho_2Z_shift(c[i] + shift_f[i], s)))
        log_weights.append(lw)
    m = max(log_weights)
    return [float(exp(w - m)) for w in log_weights]


def sample_coord_2Z_coset(alpha, s):
    sigma = float(s) / (2 * float(sqrt(2 * pi)))
    center = -QQ(alpha) / 2
    z = DGaussZ(sigma=sigma, c=center)()
    return ZZ(2 * z) + QQ(alpha)


def sample_shifted_E8(s, shift):
    if len(shift) != 8:
        raise ValueError("shift must have length 8")

    weights = codeword_weights_shifted_E8(s, shift)
    idx = random.choices(range(len(CODEWORD_TUPLES)), weights=weights, k=1)[0]
    c = CODEWORD_TUPLES[idx]
    x = [sample_coord_2Z_coset(c[i] + shift[i], s) for i in range(8)]
    return vector(QQ, x), c


def sample_coset_rep_plus_2E8(s, rep):
    # Bootstrap identity: D_{rep + 2E8, s} = 2 * D_{E8 + rep/2, s/2}.
    shift = [QQ(rep[i]) / 2 for i in range(8)]
    y, _ = sample_shifted_E8(s / 2, shift)
    return vector(QQ, [2 * yi for yi in y])


def in_E8(v):
    if len(v) != 8:
        return False
    for x in v:
        if x not in ZZ:
            return False
    parity = tuple(int(ZZ(x) % 2) for x in v)
    return parity in CODEWORD_SET


def in_2E8(v):
    if len(v) != 8:
        return False
    if any(ZZ(x) % 2 != 0 for x in v):
        return False
    half = vector(ZZ, [ZZ(x) // 2 for x in v])
    return in_E8(half)


def in_rep_plus_2E8(v, rep):
    if len(v) != len(rep):
        return False
    diff = vector(ZZ, [ZZ(v[i] - rep[i]) for i in range(8)])
    return in_2E8(diff)


def split_e8_blocks(v):
    if len(v) % 8 != 0:
        raise ValueError("vector length must be a multiple of 8")
    return [vector(v.base_ring(), list(v[8 * i:8 * (i + 1)])) for i in range(len(v) // 8)]


def sample_direct_sum_coset_rep_plus_2E8(s, coset_rep):
    blocks = split_e8_blocks(coset_rep)
    samples = [sample_coset_rep_plus_2E8(s, block) for block in blocks]
    out = []
    for sample in samples:
        out.extend(sample)
    return vector(QQ, out)


def in_direct_sum_E8(v):
    return all(in_E8(block) for block in split_e8_blocks(v))


def in_direct_sum_rep_plus_2E8(v, coset_rep):
    if len(v) != len(coset_rep):
        return False
    v_blocks = split_e8_blocks(v)
    rep_blocks = split_e8_blocks(coset_rep)
    return all(in_rep_plus_2E8(vb, rb) for vb, rb in zip(v_blocks, rep_blocks))


def lattice_point_to_basis_coefficients(v, basis):
    v_q = vector(QQ, v)
    coeffs_q = basis.transpose().solve_right(v_q.column()).column(0)
    if any(c not in ZZ for c in coeffs_q):
        raise ValueError("lattice point does not have integral coordinates in the chosen basis")
    return vector(ZZ, [ZZ(c) for c in coeffs_q])


In [ ]:
def _normalize_binary_message(msg_bits):
    if isinstance(msg_bits, str):
        if any(ch not in "01" for ch in msg_bits):
            raise ValueError("bitstring must only contain '0' and '1'")
        return msg_bits

    try:
        bits = []
        for b in msg_bits:
            if int(b) not in (0, 1):
                raise ValueError
            bits.append(str(int(b)))
        return "".join(bits)
    except Exception as exc:
        raise ValueError("msg_bits must be a bitstring or iterable of bits") from exc


def _pack_bitstring(bitstring):
    bitlen = len(bitstring)
    if bitlen == 0:
        return 0, b""
    pad = (-bitlen) % 8
    padded = bitstring + ("0" * pad)
    value = int(padded, 2)
    return bitlen, value.to_bytes(len(padded) // 8, "big")


def _normalize_salt(salt):
    if isinstance(salt, bytes):
        return salt
    if isinstance(salt, str):
        s = salt[2:] if salt.startswith("0x") else salt
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("salt hex string is invalid") from exc
    raise ValueError("salt must be bytes or hex string")


def _normalize_pubkey(pubkey_bytes):
    if pubkey_bytes is None:
        return b""
    if isinstance(pubkey_bytes, bytes):
        return pubkey_bytes
    if isinstance(pubkey_bytes, str):
        s = pubkey_bytes[2:] if pubkey_bytes.startswith("0x") else pubkey_bytes
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("pubkey hex string is invalid") from exc
    raise ValueError("pubkey_bytes must be None, bytes, or hex string")


def build_hash_transcript(msg_bits, salt, pubkey_bytes=None, domain_sep=b"E8-hawk-binary-challenge-v1"):
    bitstring = _normalize_binary_message(msg_bits)
    bitlen, packed = _pack_bitstring(bitstring)
    salt_b = _normalize_salt(salt)
    pub_b = _normalize_pubkey(pubkey_bytes)
    transcript = (
        domain_sep
        + len(pub_b).to_bytes(4, "big") + pub_b
        + len(salt_b).to_bytes(2, "big") + salt_b
        + bitlen.to_bytes(8, "big") + packed
    )
    return transcript, salt_b


def _shake_256_bits(transcript, bit_count):
    if bit_count <= 0:
        raise ValueError("bit_count must be positive")
    digest = hashlib.shake_256(transcript).digest((bit_count + 7) // 8)
    bits = []
    for byte in digest:
        for shift in range(7, -1, -1):
            bits.append(ZZ((byte >> shift) & 1))
            if len(bits) == bit_count:
                return vector(ZZ, bits)
    raise RuntimeError("not enough XOF output")


def hash_to_binary_challenge(msg_bits, salt, pubkey_bytes=None, k=1):
    basis = direct_sum_e8_basis(k)
    transcript, salt_b = build_hash_transcript(msg_bits, salt, pubkey_bytes=pubkey_bytes)
    h = _shake_256_bits(transcript, basis.nrows())
    return h, basis, salt_b


def challenge_to_lattice_shift(h, basis):
    h_vec = vector(ZZ, h)
    if len(h_vec) != basis.nrows():
        raise ValueError("challenge length must match the number of basis rows")
    if any(bit not in (ZZ(0), ZZ(1)) for bit in h_vec):
        raise ValueError("challenge h must be binary")
    coset_rep = vector(ZZ, h_vec * basis)
    half_shift = vector(QQ, [QQ(x) / 2 for x in coset_rep])
    return coset_rep, half_shift


def hash_message_to_hawk_shift(msg_bits, salt=None, pubkey_bytes=None, k=1):
    if salt is None:
        salt_b = os.urandom(16)
    else:
        salt_b = _normalize_salt(salt)

    h, basis, salt_b = hash_to_binary_challenge(
        msg_bits,
        salt_b,
        pubkey_bytes=pubkey_bytes,
        k=k,
    )
    coset_rep, half_shift = challenge_to_lattice_shift(h, basis)
    return {
        "salt": salt_b,
        "k": k,
        "b": 2,
        "basis": basis,
        "h": h,
        "coset_rep": coset_rep,
        "half_shift": half_shift,
        "challenge_is_binary": all(bit in (ZZ(0), ZZ(1)) for bit in h),
        "half_shift_doubles_to_rep": vector(ZZ, [ZZ(2 * x) for x in half_shift]) == coset_rep,
    }


In [5]:
def hawk_style_e8_sign(msg_bits, sigma, salt=None, k=1, pubkey_bytes=None):
    shift_data = hash_message_to_hawk_shift(
        msg_bits,
        salt=salt,
        pubkey_bytes=pubkey_bytes,
        k=k,
    )
    h = shift_data["h"]
    basis = shift_data["basis"]
    coset_rep = shift_data["coset_rep"]
    half_shift = shift_data["half_shift"]
    bootstrap_sigma = 2 * sigma

    bootstrap_sample = sample_direct_sum_coset_rep_plus_2E8(bootstrap_sigma, coset_rep)
    if not in_direct_sum_rep_plus_2E8(bootstrap_sample, coset_rep):
        raise RuntimeError("sampler output does not lie in h * B_E8 + 2E8^k")

    shifted_sample = vector(QQ, [QQ(x) / 2 for x in bootstrap_sample])
    shifted_difference = shifted_sample - half_shift
    if not in_direct_sum_E8(shifted_difference):
        raise RuntimeError("halved sampler output does not lie in h * B_E8 / 2 + E8^k")

    w = lattice_point_to_basis_coefficients(bootstrap_sample, basis)
    if any((w[i] - h[i]) % 2 != 0 for i in range(len(h))):
        raise RuntimeError("sample coordinates are not congruent to h modulo 2")
    s_vec = vector(ZZ, [(h[i] - w[i]) // 2 for i in range(len(h))])

    return {
        "message_bits": _normalize_binary_message(msg_bits),
        "pubkey_bytes": _normalize_pubkey(pubkey_bytes),
        "salt": shift_data["salt"],
        "k": k,
        "b": shift_data["b"],
        "sigma": sigma,
        "bootstrap_sigma": bootstrap_sigma,
        "h": h,
        "coset_rep": coset_rep,
        "half_shift": half_shift,
        "bootstrap_sample": bootstrap_sample,
        "shifted_sample": shifted_sample,
        "shifted_sample_minus_half_shift": shifted_difference,
        "w": w,
        "s_vec": s_vec,
        "challenge_is_binary": shift_data["challenge_is_binary"],
        "half_shift_doubles_to_rep": shift_data["half_shift_doubles_to_rep"],
        "sample_lies_in_coset": in_direct_sum_rep_plus_2E8(bootstrap_sample, coset_rep),
        "shifted_sample_lies_in_half_shift_plus_E8": in_direct_sum_E8(shifted_difference),
        "w_minus_h_even": all((w[i] - h[i]) % 2 == 0 for i in range(len(h))),
        "reconstructs_w_from_signature": h - 2 * s_vec == w,
    }


In [6]:
message = "10100100101101100001"
fixed_salt = bytes.fromhex("00112233445566778899aabbccddeeff")

# k = 64 gives ambient dimension 512, matches HAWK's n = 256 dimension.
k = 64
sigma = 2.2
pubkey_bytes = public_key_gram_descriptor(k)

signature = hawk_style_e8_sign(message, sigma=sigma, salt=fixed_salt, k=k, pubkey_bytes=pubkey_bytes)
print(f"parameters: k={signature['k']}, dimension={len(signature['coset_rep'])}, b={signature['b']}, sigma={signature['sigma']}")
print("checks:", {
    "binary_h": signature["challenge_is_binary"],
    "half_shift": signature["half_shift_doubles_to_rep"],
    "coset_sample": signature["sample_lies_in_coset"],
    "half_shift_sample": signature["shifted_sample_lies_in_half_shift_plus_E8"],
    "w_equals_h_minus_2s": signature["reconstructs_w_from_signature"],
})

signature_2 = hawk_style_e8_sign(message, sigma=sigma, salt=fixed_salt, k=k, pubkey_bytes=pubkey_bytes)
print("fixed transcript gives same h:", signature["h"] == signature_2["h"])
print("sampling is randomized:", signature["shifted_sample"] != signature_2["shifted_sample"])


parameters: k=64, dimension=512, b=2, sigma=2.2
checks: {'binary_h': True, 'half_shift': True, 'coset_sample': True, 'half_shift_sample': True, 'w_equals_h_minus_2s': True}
fixed transcript gives same h: True
sampling is randomized: True
